# Public Transport Travel Times

Calculate travel times in Public Transport (PT) between two sets of points given a pedestrian network and one or more GTFSs with PT supply (stops, lines, schedules, etc.).

1) Install the needed libraries (`r5py` and `Java` requirement):

In [1]:
!apt-get update -qq
!apt-get install -y openjdk-21-jdk
!pip install r5py

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-21-jdk is already the newest version (21.0.10+7-1~22.04).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


2) Download the needed data

2.1.) Download the pedestrian network (access to stations)

In [3]:
!wget -O madrid.osm.pbf https://download.geofabrik.de/europe/spain/madrid-latest.osm.pbf

--2026-05-14 12:06:46--  https://download.geofabrik.de/europe/spain/madrid-latest.osm.pbf
Resolving download.geofabrik.de (download.geofabrik.de)... 95.217.45.61, 95.217.63.98, 95.216.245.185, ...
Connecting to download.geofabrik.de (download.geofabrik.de)|95.217.45.61|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://download.geofabrik.de/europe/spain/madrid-260512.osm.pbf [following]
--2026-05-14 12:06:47--  https://download.geofabrik.de/europe/spain/madrid-260512.osm.pbf
Reusing existing connection to download.geofabrik.de:443.
HTTP request sent, awaiting response... 200 OK
Length: 83128687 (79M) [application/octet-stream]
Saving to: ‘madrid.osm.pbf’

madrid.osm.pbf      100%[===================>]  79.28M  13.8MB/s    in 6.7s    

2026-05-14 12:06:54 (11.9 MB/s) - ‘madrid.osm.pbf’ saved [83128687/83128687]



2.2.) Download the GTFSs (PT lines, stops and schedules)

We can download as many GTFSs as we need (local bus, metro, interurban bus...).

Here, we only download EMT local bus supply as an example and save it as `gtfs_emt.zip`:

In [4]:
!wget -O gtfs_emt.zip "https://datos.emtmadrid.es/dataset/9b23259a-4491-494b-9695-36a7709b2c12/resource/3cba2058-9833-422c-a704-bf992d31d2ee/download/gtfs_emt.zip"

--2026-05-14 12:06:54--  https://datos.emtmadrid.es/dataset/9b23259a-4491-494b-9695-36a7709b2c12/resource/3cba2058-9833-422c-a704-bf992d31d2ee/download/gtfs_emt.zip
Resolving datos.emtmadrid.es (datos.emtmadrid.es)... 195.57.112.136
Connecting to datos.emtmadrid.es (datos.emtmadrid.es)|195.57.112.136|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21039903 (20M) [application/zip]
Saving to: ‘gtfs_emt.zip’

gtfs_emt.zip        100%[===================>]  20.06M  6.41MB/s    in 3.1s    

2026-05-14 12:06:59 (6.41 MB/s) - ‘gtfs_emt.zip’ saved [21039903/21039903]



3) Create the origin and destination point datasets

This is just an example with 3 points in Atocha, Sol and Moncloa as origins, and 2 points as destinations (Hospital Gregorio Marañón and Hospital Clínico).

In [5]:
import geopandas as gpd
import pandas as pd

origins_df = pd.DataFrame(
    {
        "id": ["atocha", "sol", "moncloa"],
        "lon": [-3.6896, -3.7038, -3.7194],
        "lat": [40.4066, 40.4169, 40.4352],
    }
)

destinations_df = pd.DataFrame(
    {
        "id": ["hospital_gregorio_maranon", "hospital_clinico"],
        "lon": [-3.6715, -3.7197],
        "lat": [40.4193, 40.4403],
    },
)

origins_gdf = gpd.GeoDataFrame(
    origins_df,
    geometry=gpd.points_from_xy(origins_df["lon"], origins_df["lat"]),
    crs="EPSG:4326",
)

destinations_gdf = gpd.GeoDataFrame(
    destinations_df,
    geometry=gpd.points_from_xy(destinations_df["lon"], destinations_df["lat"]),
    crs="EPSG:4326",
)

4) Calculate the trip duration matrix

In [6]:
import datetime
import geopandas as gpd
import r5py

transport_network = r5py.TransportNetwork(
    "madrid.osm.pbf",
    ["gtfs_emt.zip"],   # we can add as many GTFS as we want here to this list, with the names that we have given to each when we downloaded them
)

matrix = r5py.TravelTimeMatrix(
    transport_network,
    origins=origins_gdf,
    destinations=destinations_gdf,
    transport_modes=[r5py.TransportMode.TRANSIT],
    departure=datetime.datetime(2026, 5, 14, 8, 0)
)

matrix

,from_id,to_id,travel_time
0,atocha,hospital_gregorio_maranon,20
1,atocha,hospital_clinico,37
2,sol,hospital_gregorio_maranon,31
3,sol,hospital_clinico,29
4,moncloa,hospital_gregorio_maranon,38
5,moncloa,hospital_clinico,11
